In [2]:
import requests
import json
import prettytable
headers = {'Content-type': 'application/json'}
data = json.dumps({"seriesid": ['LASST020000000000003'],"startyear":"2010", "endyear":"2021"})
p = requests.post('https://api.bls.gov/publicAPI/v2/timeseries/data/', data=data, headers=headers)
json_data = json.loads(p.text)
for series in json_data['Results']['series']:
    x=prettytable.PrettyTable(["series id","year","period","value","footnotes" ])
    seriesId = series['seriesID']
    for item in series['data']:
        year = item['year']
        period = item['period']
        value = item['value']
        footnotes=""
        for footnote in item['footnotes']:
            if footnote:
                footnotes = footnotes + footnote['text'] + ','
        if 'M01' <= period <= 'M12':
            x.add_row([seriesId,year,period,value,footnotes[0:-1]])
    output = open(seriesId + '.txt','w')
    output.write (x.get_string())
    output.close()

ConnectionError: HTTPSConnectionPool(host='api.bls.gov', port=443): Max retries exceeded with url: /publicAPI/v2/timeseries/data/ (Caused by NewConnectionError('<urllib3.connection.VerifiedHTTPSConnection object at 0x7fcfab651c90>: Failed to establish a new connection: [Errno 8] nodename nor servname provided, or not known'))

In [1]:
from pathlib import Path
import pandas as pd
import gc

In [28]:
aux='SeriesReport'

In [30]:
aux[0:7]

'SeriesR'

In [27]:
path_unem=Path('/Users/mgor/Downloads')

In [188]:
df=pd.DataFrame()

for file in path_unem.glob('*.xlsx'):

            if file.name[0:7]=='SeriesR':
                df_aux= pd.read_excel(file, header=0)
                df_aux['ST']=df_aux.loc[4][1]
                df_aux.columns = df_aux.iloc[9]
                df_aux=df_aux.loc[10:]
                df_aux=df_aux.drop(['labor force','employment','unemployment'], axis=1)
                df_aux.columns = [*df_aux.columns[:-1], 'ST']
                df_aux=df_aux.replace({'Jan': 1, 'Feb': 2,'Mar': 3,'Apr': 4,'May': 5,'Jun': 6,'Jul': 7,'Aug': 8,'Sep': 9,'Oct': 10,'Nov': 11,'Dec': 12})
                df_aux.loc[df_aux['Period'] < 4, 'quarter'] = 1
                df_aux.loc[(df_aux['Period'] > 3) & (df_aux['Period'] < 7), 'quarter'] = 2
                df_aux.loc[(df_aux['Period'] > 6) & (df_aux['Period'] < 10), 'quarter'] = 3
                df_aux.loc[(df_aux['Period'] > 9) & (df_aux['Period'] < 13), 'quarter'] = 4
                df_aux=df_aux.drop(['Period'],axis=1)
                df_aux=df_aux.groupby(['Year', 'ST','quarter']).mean().reset_index()
                df=df.append(df_aux)
                del df_aux
                gc.collect()



In [189]:
df=df.drop(['labor force participation rate', 'employment-population ratio'],axis=1)

In [195]:
df_aux= pd.read_table('/Users/mgor/Downloads/us-state-ansi-fips.csv', delimiter=',', header=0)



In [196]:
df_aux=df_aux.rename(columns={ 'stname':'ST',' st':'ST FIPS Code'})

In [198]:
df=df.merge(df_aux, how='left', on=['ST'], indicator=True)



In [199]:
df.loc[df['ST'] == 'Puerto Rico', 'ST FIPS Code']=72

In [200]:

df=df.drop([' stusps','_merge'],axis=1)

In [201]:

df=df.rename(columns={ 'Year':'year'})

In [202]:
df=df.sort_values(['ST FIPS Code', 'year', 'quarter'])

In [204]:
df['ST FIPS Code'].unique()

array([ 1.,  2.,  4.,  5.,  6.,  8.,  9., 10., 11., 12., 13., 15., 16.,
       17., 18., 19., 20., 21., 22., 23., 24., 25., 26., 27., 28., 29.,
       30., 31., 32., 33., 34., 35., 36., 37., 38., 39., 40., 41., 42.,
       44., 45., 46., 47., 48., 49., 50., 51., 53., 54., 55., 56., 72.])

In [206]:
df.to_csv('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/Labor/unem_state.csv.zip', compression='zip', index=False)



In [211]:
df1=pd.read_stata('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/Labor/compustat_bg_names_dedupl.dta')

In [225]:
df1=df1.rename(columns={ 'headquarter_state_compustat':'ST','name_compustat':'employer'})

In [218]:
df=df.merge(df1, how='left', on=['ST'], indicator=True)



In [228]:
df.groupby(['ST','ST FIPS Code','gvkey','employer','year','quarter']).mean().reset_index()

,ST,ST FIPS Code,gvkey,employer,year,quarter,unemployment rate,compustat_year1,compustat_year2,subsidiary_compustat
0,Alabama,1.0,005210,GOLDEN ENTERPRISES,2010.0,1.0,11.000000,1968.0,2015.0,0.00
1,Alabama,1.0,005210,GOLDEN ENTERPRISES,2010.0,2.0,10.300000,1968.0,2015.0,0.00
2,Alabama,1.0,005210,GOLDEN ENTERPRISES,2010.0,3.0,10.066667,1968.0,2015.0,0.00
3,Alabama,1.0,005210,GOLDEN ENTERPRISES,2010.0,4.0,10.133333,1968.0,2015.0,0.00
4,Alabama,1.0,005210,GOLDEN ENTERPRISES,2011.0,1.0,10.000000,1968.0,2015.0,0.00
...,...,...,...,...,...,...,...,...,...,...
178342,Wyoming,56.0,180635,CLOUD PEAK ENERGY INC,2021.0,3.0,4.333333,2004.0,2018.0,0.75
178343,Wyoming,56.0,180635,CLOUD PEAK ENERGY INC,2021.0,4.0,4.000000,2004.0,2018.0,0.75
178344,Wyoming,56.0,180635,CLOUD PEAK ENERGY INC,2022.0,1.0,3.600000,2004.0,2018.0,0.75
178345,Wyoming,56.0,180635,CLOUD PEAK ENERGY INC,2022.0,2.0,3.200000,2004.0,2018.0,0.75


In [229]:
df=df.drop(['compustat_year1','compustat_year2','subsidiary_compustat'],axis=1)

In [231]:
df=df.drop(['subsidiary_name_compustat','_merge'],axis=1)

In [236]:
df.to_csv('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/Labor/unem_state_comp.csv.zip', compression='zip', index=False)



In [239]:
df.to_stata('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/Labor/unem_state_comp.dta')



/opt/anaconda3/lib/python3.7/site-packages/pandas/io/stata.py:2397: InvalidColumnName: 
Not all pandas column names were valid Stata variable names.
The following replacements have been made:

    unemployment rate   ->   unemployment_rate
    ST FIPS Code   ->   ST_FIPS_Code

If this is not what you expect, please make sure you have Stata-compliant
column names in your DataFrame (strings only, max 32 characters, only
alphanumerics and underscores, no Stata reserved words)

  warnings.warn(ws, InvalidColumnName)


In [1]:
df_aux=pd.read_stata('/Users/mgor/Documents/Strategy/2YP/Data/data_merged/Merged_main/panel_main.dta',encoding='latin')  

NameError: name 'pd' is not defined

In [47]:
df_aux.dtypes

index                    int32
year                     int32
quarter                float64
employer                object
gvkey                    int32
                        ...   
softw_skill_growth     float64
ST_FIPS_Code           float64
month                  float64
Unemployment_Rate      float64
_merge                category
Length: 195, dtype: object

In [3]:
df_unem=pd.read_stata('/Users/mgor/Documents/Strategy/2YP/Data/data_csv/Labor/unem_state_comp.dta')

In [31]:
df_aux1=df_aux[1:100]

In [32]:
df_aux1=df_aux1.drop(['_merge'],axis=1)

In [21]:
df_unem1=df_unem.rename(columns={ 'unemployment_rate':'unem_ST'})

In [46]:
df_unem1

,year,quarter,unem_ST,ST_FIPS_Code
0,2010.0,1.0,11.000000,1.0
1,2010.0,1.0,11.000000,1.0
2,2010.0,1.0,11.000000,1.0
3,2010.0,1.0,11.000000,1.0
4,2010.0,1.0,11.000000,1.0
...,...,...,...,...
1526680,2021.0,3.0,8.100000,72.0
1526681,2021.0,4.0,7.500000,72.0
1526682,2022.0,1.0,6.800000,72.0
1526683,2022.0,2.0,6.233333,72.0


In [25]:
df_unem1=df_unem1.drop(['index','ST','name_bgt','employer','gvkey'],axis=1)

In [33]:
df_aux1=df_aux1.merge(df_unem1, how='left', on=['year','quarter','ST_FIPS_Code'], indicator=True)

In [38]:
df_unem=df_unem.rename(columns={ 'unem_ST':'unem_head'})
df_unem=df_unem.drop(['ST','ST_FIPS_Code','gvkey','name_bgt','index'],axis=1)
df_aux1=df_aux1.rename(columns={ '_merge':'merge_1'})

In [39]:
df_aux1=df_aux1.merge(df_unem, how='left', on=['year','quarter','employer'], indicator=True)